# Data Cleaning and Panel Construction

This notebook constructs the final country–year panel dataset used in the analysis.

Goals:
- Clean WHO dataset
- Clean World Bank dataset
- Pivot WDI dataset
- Harmonize countries
- Merge datasets
- Inspect missingness
- Apply imputation rules

The output of this notebook is the cleaned analytical dataset saved to `data/processed/`.


### Verifying Project Root
(this step is neccessary given we will have to work with nested directories)

In [1]:
import sys
from pathlib import Path

# find project root (one level above notebooks/)
PROJECT_ROOT = Path().resolve().parents[0]
sys.path.append(str(PROJECT_ROOT / "src"))

print("Project root:", PROJECT_ROOT)

Project root: /Users/rnd4impact/Desktop/RND4IMPACT Projects/life-expectancy-analysis


### Imports

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

from data.data_loading import load_raw, standardize, merge
from data.data_preprocessing import (
    clean_who,
    clean_wb,
    pivot_wdi,
    missingness_table,
    missingness_by_year,
    missingness_by_country,
    impute_panel,
)

### Used Paths

In [3]:
RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")
PROCESSED.mkdir(exist_ok=True)

# Loading standardized datasets

In [4]:
who_path = Path("../data/raw/world_health_organization/who.csv")  
who_raw = load_raw("who", path=who_path)
who = standardize(who_raw, "who")

wb_path = Path("../data/raw/world_bank/wb.csv")  
wb_raw = load_raw("wb", path=wb_path)
wb = standardize(wb_raw, "wb")

wdi_path = Path("../data/raw/world_bank/wdi.csv")  
wdi_raw = load_raw("wdi", path=wdi_path)
wdi = standardize(wdi_raw, "wdi")

## Dataset Cleaning

### WHO

In [6]:
who_clean, who_summary = clean_who(
    who,
    min_years_per_country=10,
    feature_missingness_drop_threshold=0.8
)

who_summary

{'input_rows': 2938,
 'input_cols': 22,
 'duplicate_policy': 'keep_first',
 'min_years_per_country': 10,
 'drop_missing_target': True,
 'feature_missingness_drop_threshold': 0.8,
 'duplicate_rows': 0,
 'rows_after_dedup': 2938,
 'life_expectancy_invalid_count': 0,
 'immunization_cols_clipped': ['hepatitis_b',
  'measles',
  'polio',
  'diphtheria'],
 'rows_dropped_missing_target': 10,
 'rows_dropped_low_coverage': 0,
 'countries_kept': 183,
 'dropped_feature_cols': [],
 'n_dropped_feature_cols': 0,
 'output_rows': 2928,
 'output_cols': 22,
 'year_min': 2000,
 'year_max': 2015}

### WB

In [9]:
wb_clean, wb_summary = clean_wb(
    wb,
    winsorize_cols=["GDP", "CO2"],
    log_transform_cols=["GDP"],
    feature_missingness_drop_threshold=0.7
)

wb_summary

{'input_rows': 3306,
 'input_cols': 16,
 'feature_missingness_drop_threshold': 0.7,
 'winsorize_cols': ['GDP', 'CO2'],
 'log_transform_cols': ['GDP'],
 'dropped_feature_cols': ['corruption'],
 'n_dropped_feature_cols': 1,
 'duplicate_rows': 0,
 'output_rows': 3306,
 'output_cols': 15,
 'year_min': 2001,
 'year_max': 2019}

### WDI Pivotting

In [10]:
wdi_panel, wdi_summary = pivot_wdi(
    wdi,
    indicator_codes=["SP.DYN.LE00.IN"],
    year_min=1950
)

wdi_summary

{'input_rows': 266,
 'rows_after_indicator_filter': 266,
 'n_year_columns': 65,
 'indicator_codes': ['SP.DYN.LE00.IN'],
 'rows_dropped_all_nan': 0,
 'output_rows': 16926,
 'output_cols': 3}

In [11]:
wdi_panel.head()

,country,year,SP.DYN.LE00.IN
0,Afghanistan,1960,32.799
1,Afghanistan,1961,33.291
2,Afghanistan,1962,33.757
3,Afghanistan,1963,34.201
4,Afghanistan,1964,34.673


## WHO + WB Merging

In [21]:
panel, merge_diag = merge(
    who_clean,
    wb_clean,
    how="inner"
)

merge_diag

{'who_rows': 2928,
 'wb_rows': 3306,
 'how': 'inner',
 'on': ('country', 'year'),
 'merged_rows': 2265,
 'who_unique_keys': 2928,
 'wb_unique_keys': 3306,
 'merged_unique_keys': 2265,
 'keys_lost_from_who': 663,
 'keys_lost_from_wb': 1041}

In [15]:
bigger_panel, merge_diag = merge(
    who_clean,
    wb_clean,
    how="outer"
)

merge_diag

{'who_rows': 2928,
 'wb_rows': 3306,
 'how': 'outer',
 'on': ('country', 'year'),
 'merged_rows': 3969,
 'who_unique_keys': 2928,
 'wb_unique_keys': 3306,
 'merged_unique_keys': 3969,
 'keys_lost_from_who': 0,
 'keys_lost_from_wb': 0}

## Missingness Analysis

In [37]:
panel.head()

,country,year,status,life_expectancy,adult_mortality,infant_deaths,alcohol,percentage_expenditure,hepatitis_b,measles,bmi,under_five_deaths,polio,total_expenditure,diphtheria,hiv_aids,gdp,population,thinness_1_19_years,thinness_5_9_years,income_composition_of_resources,schooling,country_code,region,income_group,life_expectancy_wb,undernourishment,co2,health_expenditure_percent,education_expenditure_percent,unemployment,sanitation,injuries,communicable,noncommunicable_disease
0,Afghanistan,2015,Developing,65.0,263.0,62,0.01,71.279624,65.0,100,19.1,83,6.0,8.16,65.0,0.1,584.259210,33736494.0,17.2,17.3,0.479,10.1,AFG,South Asia,Low income,63.377,21.5,5949.999809,10.105348,3.25580,11.127,NaN,3673696.62,6528888.62,6988545.28
1,Afghanistan,2014,Developing,59.9,271.0,64,0.01,73.523582,62.0,100,18.6,86,58.0,8.18,62.0,0.1,612.696514,327582.0,17.5,17.5,0.476,10.0,AFG,South Asia,Low income,62.966,20.7,4880.000114,9.528871,3.69522,11.142,NaN,3267937.78,6649335.87,6900348.40
2,Afghanistan,2013,Developing,59.9,268.0,66,0.01,73.219243,64.0,100,18.1,89,62.0,8.13,64.0,0.1,631.744976,31731688.0,17.7,17.7,0.470,9.9,AFG,South Asia,Low income,62.525,20.7,5989.999771,8.805941,3.45446,11.193,NaN,2807904.86,6813189.19,6799914.37
3,Afghanistan,2012,Developing,59.5,272.0,69,0.01,78.184215,67.0,100,17.6,93,67.0,8.52,67.0,0.1,669.959000,3696958.0,17.9,18.0,0.463,9.8,AFG,South Asia,Low income,62.054,21.1,8079.999924,7.897176,3.32000,11.341,NaN,2715550.23,7036448.02,6640268.93
4,Afghanistan,2011,Developing,59.2,275.0,71,0.01,7.097109,68.0,100,17.2,97,68.0,7.87,68.0,0.1,63.537231,2978599.0,18.2,18.2,0.454,9.5,AFG,South Asia,Low income,61.553,20.2,8930.000305,8.561907,3.46201,11.054,NaN,2540038.63,7181018.86,6539124.12


In [23]:
miss_table = missingness_table(panel)
miss_table

,column,missing_fraction,missing_count,non_missing_count
0,sanitation,0.3536,801,1464
1,education_expenditure_percent,0.3166,717,1548
2,hepatitis_b,0.1788,405,1860
3,undernourishment,0.1457,330,1935
4,population,0.0945,214,2051
5,total_expenditure,0.0777,176,2089
6,alcohol,0.0711,161,2104
7,unemployment,0.0265,60,2205
8,health_expenditure_percent,0.0247,56,2209
9,gdp,0.0137,31,2234


Missingness by year:

In [33]:
miss_year = missingness_by_year(panel, cols=["life_expectancy_wb"])
miss_year

,year,n_rows,life_expectancy_wb_missing_fraction
0,2001,151,0.0
1,2002,151,0.0
2,2003,151,0.0
3,2004,151,0.0
4,2005,151,0.0
5,2006,151,0.0
6,2007,151,0.0
7,2008,151,0.0
8,2009,151,0.0
9,2010,151,0.0


Missingness by country:

In [36]:
miss_country = missingness_by_country(panel, cols=["life_expectancy_wb"])
miss_country

,country,n_rows,life_expectancy_wb_missing_fraction
0,Afghanistan,15,0.0
1,Mozambique,15,0.0
2,Namibia,15,0.0
3,Nepal,15,0.0
4,Netherlands,15,0.0
5,New Zealand,15,0.0
6,Nicaragua,15,0.0
7,Niger,15,0.0
8,Nigeria,15,0.0
9,Norway,15,0.0


## Imputation

In [38]:
panel["life_expectancy_final"] = panel[["life_expectancy", "life_expectancy_wb"]].mean(axis=1)
panel.head()

,country,year,status,life_expectancy,adult_mortality,infant_deaths,alcohol,percentage_expenditure,hepatitis_b,measles,bmi,under_five_deaths,polio,total_expenditure,diphtheria,hiv_aids,gdp,population,thinness_1_19_years,thinness_5_9_years,income_composition_of_resources,schooling,country_code,region,income_group,life_expectancy_wb,undernourishment,co2,health_expenditure_percent,education_expenditure_percent,unemployment,sanitation,injuries,communicable,noncommunicable_disease,life_expectancy_final
0,Afghanistan,2015,Developing,65.0,263.0,62,0.01,71.279624,65.0,100,19.1,83,6.0,8.16,65.0,0.1,584.259210,33736494.0,17.2,17.3,0.479,10.1,AFG,South Asia,Low income,63.377,21.5,5949.999809,10.105348,3.25580,11.127,NaN,3673696.62,6528888.62,6988545.28,64.1885
1,Afghanistan,2014,Developing,59.9,271.0,64,0.01,73.523582,62.0,100,18.6,86,58.0,8.18,62.0,0.1,612.696514,327582.0,17.5,17.5,0.476,10.0,AFG,South Asia,Low income,62.966,20.7,4880.000114,9.528871,3.69522,11.142,NaN,3267937.78,6649335.87,6900348.40,61.4330
2,Afghanistan,2013,Developing,59.9,268.0,66,0.01,73.219243,64.0,100,18.1,89,62.0,8.13,64.0,0.1,631.744976,31731688.0,17.7,17.7,0.470,9.9,AFG,South Asia,Low income,62.525,20.7,5989.999771,8.805941,3.45446,11.193,NaN,2807904.86,6813189.19,6799914.37,61.2125
3,Afghanistan,2012,Developing,59.5,272.0,69,0.01,78.184215,67.0,100,17.6,93,67.0,8.52,67.0,0.1,669.959000,3696958.0,17.9,18.0,0.463,9.8,AFG,South Asia,Low income,62.054,21.1,8079.999924,7.897176,3.32000,11.341,NaN,2715550.23,7036448.02,6640268.93,60.7770
4,Afghanistan,2011,Developing,59.2,275.0,71,0.01,7.097109,68.0,100,17.2,97,68.0,7.87,68.0,0.1,63.537231,2978599.0,18.2,18.2,0.454,9.5,AFG,South Asia,Low income,61.553,20.2,8930.000305,8.561907,3.46201,11.054,NaN,2540038.63,7181018.86,6539124.12,60.3765


In [40]:
panel_imputed, impute_summary = impute_panel(
    panel,
    interpolate_cols=["gdp", "health_expenditure_percent", "sanitation"],
    group_cols=["region", "income_group"],
    group_median_cols=["education_expenditure_percent"],
    median_impute_cols=["unemployment", "corruption"],
)

impute_summary


{'input_rows': 2265,
 'interpolate_cols': ['gdp', 'health_expenditure_percent', 'sanitation'],
 'median_impute_cols': ['unemployment', 'corruption'],
 'group_cols': ['region', 'income_group'],
 'group_median_cols': ['education_expenditure_percent'],
 'interpolate_filled_counts': {'gdp': 11,
  'health_expenditure_percent': 9,
  'sanitation': 3},
 'group_median_filled_counts': {'education_expenditure_percent': 717},
 'median_filled_counts': {'unemployment': 60},
 'output_rows': 2265,
 'target_missing_count_after': 0}

In [41]:
panel_imputed.to_csv(PROCESSED / "main_dataset.csv", index=False)